# Unit 06 · A controlled experiment

Unit 05 trained one network once. Now we ask a question and answer it in a way someone else can trust:
**how much is the second X-ray worth?** And the proposal's second one: **how much does more training data help?**
The whole unit is about honesty: change one thing, repeat with several seeds, compare patient by patient, and say
"within noise" when that's what the numbers say.

| Part | After video 12 | What you build | Suggested driver |
|---|---|---|---|
| A | one variable at a time | nothing to write: read the design, run the smoke test here, the real run on Kaggle | both |
| B | variance; 3 seeds, mean ± std; pairing | the seed spread, per-patient pairs (comparison 1: 1 vs 2 views) | **Adir** (lead) / Noa |
| C | reading a results table honestly | the bootstrap interval, the decision rule, the table (comparison 2: training-set size) | **Noa** (lead) / Adir |

**Needs:** units 02–05 moved into `src/xray3d` (`metrics.evaluate`; `nets.XrayTo3D`; `losses.bce_dice_loss`;
`training.train_step`, `training.validation_dice`), the processed dataset v1, `splits/split_v1.json` and unit 03's
`results/baselines.csv` (for the table). Locally everything runs on the CPU in about 10 minutes, with a **smoke test
whose numbers mean nothing**; the real experiment (9 runs × 4,000 steps) runs on a Kaggle GPU with the same code.

In [ ]:
# --- setup (written by Claude, no need to change it): the same cell works locally, on Colab and on Kaggle ---
import os, sys, subprocess
from pathlib import Path

UNIT = "06-experiment"
REPO_URL = "https://github.com/AdirBuskila/pelvis-xray-3d"
IN_CLOUD = "google.colab" in sys.modules or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_CLOUD:
    root = Path("/content" if "google.colab" in sys.modules else "/kaggle/working") / "pelvis-xray-3d"
    if not root.exists():  # Kaggle: turn Internet on in the notebook settings first
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
else:  # local: the repo is the first folder above this notebook that has a pyproject.toml
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root / "units" / UNIT)
for p in (root / "src", Path.cwd()):  # import xray3d (even before `pip install -e .`) and this unit's checks
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import xray3d
from xray3d import data, env, plotting
import test_unit as T

env.summary()

In [ ]:
import torch
from xray3d import metrics, nets, torchdata, trainer
import run_experiment as RX            # Claude's run script, in this folder (also a command-line tool)

need = [(metrics, "evaluate"), (nets, "XrayTo3D")]
try:
    from xray3d import losses, training
    need += [(losses, "bce_dice_loss"), (training, "train_step"), (training, "validation_dice")]
except ImportError as e:
    print("import problem:", e)
missing = [f"{m.__name__}.{f}" for m, f in need if not hasattr(m, f)]
print("units 02-05 functions missing from src/xray3d:", missing or "none, good")
sp = torchdata.split_subjects()
print({k: len(v) for k, v in sp.items()}, "subjects in train / val / test")

---
## Part A · The design (after video 12, first part)

**Change one variable.** Every run uses the trainer's defaults (unit 05): the same network code, 4,000 steps, batch 8,
Adam at 1e-3, the same augmentation, the same split, best checkpoint chosen on **validation**, scored once on **test**.
An arm differs from another in exactly one key:

In [ ]:
print(f"{'arm':<12}{'n_views':>8}{'train_frac':>11}   everything else: trainer.DEFAULTS")
for arm, kv in RX.ARMS.items():
    print(f"{arm:<12}{kv['n_views']:>8}{kv['train_frac']:>11}")
print("\ncomparisons (A - B):", {k: f"{a} - {b}" for k, (a, b) in RX.COMPARISONS.items()})
print("seeds:", RX.SEEDS, "| steps:", RX.ITERS, "| fixed:", {k: v for k, v in trainer.DEFAULTS.items() if k not in ("n_views", "train_frac", "seed")})

n_train = len(sp["train"])
print(f"\ntraining subjects: 100 % = {n_train}, 25 % = {len(torchdata.subset(sp['train'], 0.25))} (nested: the 25 % set is inside the full set)")
for v in (1, 2):
    print(f"XrayTo3D(n_views={v}): {sum(p.numel() for p in nets.XrayTo3D(n_views=v).parameters()):,} parameters")

Two comparisons of two arms with three seeds would be 12 runs. But "2 views, 100 % of the training set" is the
**same configuration** in both, so the same three runs serve both comparisons: **9 runs**.

**One thing is not equal:** the 2-view network has more parameters, because it has a second encoder for the LAT
image. Video 12 shows another way to build the 1-view arm: keep the 2-view network and feed it an all-zero LAT
image, so the parameter count is identical. We use `n_views=1` (the network of unit 04, and what unit 07 runs on real
AP frames). Keep this in mind for the explain-back: is it a confound, and which way could it push the result?

### The smoke test (local, CPU)

`RX.run_and_evaluate` trains every arm × seed through `trainer.train` (resumable: a finished run is skipped, an
unfinished one continues from `last.pt`), then loads each run's `best.pt`, predicts every **test** subject and
scores it with **your** `metrics.evaluate`. Here it runs a tiny configuration: 2 seeds, 40 steps of batch 4,
for all three arms, about 5–8 minutes on the laptop.

> ⚠️ **SMOKE TEST: the numbers are meaningless.** 40 steps teach the network almost nothing. This run only proves
> the pipeline works end to end, so the analysis below has a table to work on until the real run is back from Kaggle.

In [ ]:
smoke_cfg = {k: v for k, v in RX.SMOKE.items() if k not in ("seeds", "iters")}
print("smoke configuration:", RX.SMOKE)
SMOKE_CSV = RX.run_and_evaluate(seeds=RX.SMOKE["seeds"], iters=RX.SMOKE["iters"], smoke=True, verbose=False, **smoke_cfg)

In [ ]:
smoke_runs = [RX.run_name(a, s, "smoke") for a in RX.ARMS for s in RX.SMOKE["seeds"]]
trainer.plot_curves(smoke_runs, title="SMOKE TEST (40 steps): the curves of all six runs, numbers meaningless")
plt.show()
for r in smoke_runs:
    print(f"{r:<22} {trainer.read_log(r)['seconds'][-1]:>6.0f} s of training")

### The real run (Kaggle GPU)

The same code, with 3 seeds and 4,000 steps. It reads your functions from `src/xray3d` **of the GitHub repo**
(Kaggle clones it), so commit and push units 02–05 first.

1. On Kaggle: *Create → New Notebook → File → Import Notebook*, and give it this notebook's GitHub URL
   (`units/06-experiment/notebook.ipynb`), or upload the file.
2. *Settings*: **Accelerator → GPU T4 ×1** (or P100), **Internet → on**. *Add Input* → your dataset
   **`pelvis-xray3d-v1`** (unit 02 built it).
3. **Save Version → "Save & Run All (Commit)"**. That runs the notebook top to bottom in the background (up to 12 hours,
   no need to keep the browser open) and keeps `/kaggle/working` as the version's **Output**.
4. When it finishes, download from *Output*: `pelvis-xray-3d/results/experiment_v1.csv` (put it in the repo's
   `results/` and commit it) and, if you want the checkpoints for unit 07, the `runs/` folder (`exp-*`).

**How long?** On the CPU here one step of batch 8 takes about 2 s. On a T4 a step is roughly 0.1–0.25 s, so a
4,000-step run takes about **7–17 minutes** (1-view runs a bit less), and the 9 runs about **1–2.5 hours**, plus a few
minutes of test scoring. That fits in one session (Kaggle gives 12 hours per session, about 30 GPU hours per week).
The `seconds` column of each `log.csv` records the real time: write it in `LOG.md`.

**Splitting it across sessions** (if a session dies, or to keep each one short): run part of the seeds, e.g.
`RX.run_and_evaluate(seeds=(0,))`, Save Version, then in the next version add the previous version's **output** as an
input (*Add Input → Your Work → Notebook Output*) and set `PREV` below to its `runs` folder: the finished runs are
copied over and skipped, an unfinished one resumes from its `last.pt`, and the CSV at the end has every run.
From a terminal it's `python units/06-experiment/run_experiment.py [--seeds 0] [--restore-from DIR]`.
On the laptop this cell only prints a message.

In [ ]:
PREV = None        # e.g. "/kaggle/input/<your previous version>/runs" when continuing in a new session
if env.has_gpu():
    if PREV:
        RX.restore_runs(PREV)
    if env.where() == "colab":                 # Colab also works: runs are copied to Drive after every checkpoint
        from google.colab import drive
        drive.mount("/content/drive")
        from xray3d import dataset
        dataset.ensure_local()
    REAL_CSV = RX.run_and_evaluate(seeds=RX.SEEDS, iters=RX.ITERS)
    for a in RX.ARMS:
        for s in RX.SEEDS:
            n = RX.run_name(a, s)
            if (trainer.run_dir(n) / "log.csv").exists():
                print(f"{n:<18} {trainer.read_log(n)['seconds'][-1] / 60:5.1f} min")
else:
    print("No GPU here: the real experiment runs on Kaggle (see above). The analysis below uses the smoke test.")

**Explain-back (part A):**
1. Why does each arm change exactly one key? Name one thing that would go wrong if the 2-view runs also trained
   for 8,000 steps.
2. The 25 % arm trains for the same 4,000 steps on a quarter of the patients. So it sees each patient 4× as often.
   Is "same number of steps" or "same number of epochs" the fair comparison? What would each one measure?
3. The smoke runs and the real runs have different names (`smoke-…`, `exp-…`). Why must they never share a folder?

### Which results do we analyse?

If `results/experiment_v1.csv` exists (the real run, downloaded from Kaggle), the rest of the notebook uses it.
Otherwise it uses the smoke test, and says so on every table and figure. An **empty prediction** (no voxel above 0.5,
common after 40 smoke steps) has HD95 = inf: there is no predicted surface to measure a distance from. So a smoke
run's HD95 means can come out inf or nan. That is expected, not a bug; the table cell below says how many rows it was.

In [ ]:
REAL = env.results_root() / "experiment_v1.csv"
CSV = REAL if REAL.exists() else SMOKE_CSV
IS_SMOKE = CSV != REAL
TAG = "SMOKE TEST, numbers meaningless" if IS_SMOKE else "experiment v1"
rows = RX.load_rows(CSV)
BASE_CSV = env.results_root() / "baselines.csv"
base = RX.load_rows(BASE_CSV) if BASE_CSV.exists() else []
print(f"analysing {CSV}  ->  {TAG}")
print(f"{len(rows)} rows: arms {sorted({r['arm'] for r in rows})}, seeds {sorted({r['seed'] for r in rows})}, "
      f"{len({r['subject'] for r in rows})} test subjects")
print(f"baselines: {len(base)} rows from {BASE_CSV.name}" if base else "baselines.csv not found (run unit 03): the table will have no baselines")
print("one row:", {k: rows[0][k] for k in ("method", "subject", "arm", "seed", "n_views", "train_frac", "dice", "hd95")})

---
## Part B · Seeds, and patients in pairs (after video 12: variance) · comparison 1: 1 view vs 2 views

Two kinds of noise sit in every number of the CSV:
* **the seed** (training randomness: initial weights, data order, augmentation, dropout). Run the same arm again
  with another seed and its test Dice moves. That's what the three seeds per arm measure.
* **the patients** (which subjects happen to be in the test set). A hard pelvis is hard for every arm.

Part B separates the two: one number per **run** (averaging over patients) shows the seed spread; one number per
**patient** (averaging over seeds) is what gets paired between arms.

### TODO B1 · `mean_std`  ·  suggested driver: **Adir**

The mean and the **sample** standard deviation of a few numbers (e.g. one test Dice per seed). Return
`(mean, std)` as floats. With a single value there is no spread to estimate: return `nan` as its std.

> **Nudge:** NumPy's `std` divides by n unless you tell it otherwise. With 3 seeds, why divide by n − 1 (video 12)? Accept any iterable: `list(values)` first, so `dict.values()` and generators work too.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def mean_std(values):
    """Mean and sample standard deviation (divide by n - 1).

    values  : any iterable of numbers, e.g. [0.78, 0.80, 0.82] or some_dict.values()
    returns : (mean, std) as Python floats; std is nan when there is only one value
    """
    raise NotImplementedError("TODO B1")

In [ ]:
T.check_mean_std(mean_std)

### TODO B2 · `seed_scores`  ·  suggested driver: **Noa**

One number per **run** of an arm: for every seed, the mean of `metric` over that run's test subjects. `rows` is the
CSV as a list of dicts (`RX.load_rows`: `r["arm"]`, `r["seed"]` (int), `r["subject"]`, `r["dice"]`, ...), in no
particular order. Return `{seed: score}` with the seeds in increasing order.

> **Nudge:** keep only this arm's rows; group the values by seed (a dict of lists: `groups.setdefault(seed, []).append(value)`), then average each group.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def seed_scores(rows, arm, metric="dice"):
    """{seed: mean of `metric` over the test subjects of that run}, for one arm.

    rows   : list of dicts with keys "arm", "seed", "subject", and the metric
    returns: dict, keys sorted (0, 1, 2, ...)
    """
    raise NotImplementedError("TODO B2")

In [ ]:
T.check_seed_scores(seed_scores)

In [ ]:
ARMS = list(dict.fromkeys(r["arm"] for r in rows))
COLORS = {"ap": "#d1495b", "ap_lat": "#3b6fb6", "ap_lat_f25": "#edae49"}
fig, ax = plt.subplots(figsize=(8, 0.9 + 0.8 * len(ARMS)))
for y, arm in enumerate(ARMS):
    sc = seed_scores(rows, arm)
    m, s = mean_std(sc.values())
    ax.errorbar(m, y, xerr=s if np.isfinite(s) else 0, fmt="none", ecolor="#1d1d1d", capsize=6, lw=1.5)
    ax.plot(m, y, "|", color="#1d1d1d", ms=22, mew=2)
    ax.scatter(list(sc.values()), [y] * len(sc), s=70, color=COLORS.get(arm, "gray"), zorder=3, edgecolor="white")
    for seed, v in sc.items():
        ax.annotate(f"seed {seed}", (v, y), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8, color="#555")
    print(f"{arm:<12} test Dice per seed {[round(v, 3) for v in sc.values()]}  ->  {m:.3f} ± {s:.3f}")
ax.set_yticks(range(len(ARMS)), ARMS); ax.set_ylim(-0.6, len(ARMS) - 0.4); ax.invert_yaxis()
ax.set_xlabel("test Dice of one run (mean over the test subjects)"); ax.grid(axis="x", alpha=0.3)
ax.set_title(f"one dot per run; bar = mean ± sample std over seeds  ({TAG})")
plt.tight_layout(); plt.show()

Video 12's warning in numbers: pick one seed of each arm and compare them. Depending on which two you pick, how
different is the story? (With the smoke test the dots are random; with the real run, look again.)

### TODO B3 · `per_subject_mean`  ·  suggested driver: **Adir**

Now the other direction: one number per **patient** for an arm, the mean of `metric` over that arm's seeds.
Return `{subject: score}` with the subjects sorted.

> **Nudge:** the same grouping as B2 with another key. Ask yourselves: why average the seeds first, before pairing two arms? (What would you pair otherwise: seed 0 of one arm with seed 0 of the other? Those two runs have nothing in common.)

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def per_subject_mean(rows, arm, metric="dice"):
    """{subject: mean of `metric` over the seeds of this arm}.

    rows   : list of dicts with keys "arm", "seed", "subject", and the metric
    returns: dict, keys sorted
    """
    raise NotImplementedError("TODO B3")

In [ ]:
T.check_per_subject_mean(per_subject_mean)

### TODO B4 · `paired_differences`  ·  suggested driver: **Noa**

Pair two arms **patient by patient**: `a[s] − b[s]` for every subject `s`. Both arms must have been scored on the
same patients; if the subject sets differ, the pairing is meaningless: raise `ValueError` and say which subjects
don't match.

> **Nudge:** compare the two key sets first (`set(a) != set(b)`); then loop over the sorted subjects. Never pair by position: two dicts (or two CSVs) need not list the patients in the same order.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def paired_differences(a, b):
    """{subject: a[subject] - b[subject]}, keys sorted.

    a, b    : {subject: score}, e.g. per_subject_mean(rows, "ap_lat") and per_subject_mean(rows, "ap")
    raises  : ValueError if a and b don't hold exactly the same subjects
    """
    raise NotImplementedError("TODO B4")

In [ ]:
T.check_paired_differences(paired_differences)

In [ ]:
def paired_figure(arm_a, arm_b, metric="dice", label_a=None, label_b=None):
    """Left: one line per test patient from arm B to arm A. Right: the per-patient differences A - B, sorted."""
    a, b = per_subject_mean(rows, arm_a, metric), per_subject_mean(rows, arm_b, metric)
    d = paired_differences(a, b)
    subs = list(d)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1, 1.6]})
    for s in subs:
        up = d[s] > 0
        ax[0].plot([0, 1], [b[s], a[s]], "-o", color="#3b6fb6" if up else "#d1495b", alpha=0.75, ms=5, lw=1.5)
    ax[0].plot([0, 1], [np.mean(list(b.values())), np.mean(list(a.values()))], "-", color="#1d1d1d", lw=3.5, label="mean")
    ax[0].set_xticks([0, 1], [label_b or arm_b, label_a or arm_a]); ax[0].set_xlim(-0.3, 1.3)
    ax[0].set_ylabel(f"test {metric} (mean over seeds)"); ax[0].legend(loc="best"); ax[0].grid(axis="y", alpha=0.3)
    ax[0].set_title(f"one line per test patient (blue: {label_a or arm_a} better)")
    order = sorted(subs, key=d.get)
    ax[1].bar(range(len(order)), [d[s] for s in order], color=["#3b6fb6" if d[s] > 0 else "#d1495b" for s in order])
    ax[1].axhline(0, color="#1d1d1d", lw=1)
    ax[1].axhline(np.mean(list(d.values())), color="#1d1d1d", ls="--", lw=1.2, label=f"mean difference {np.mean(list(d.values())):+.3f}")
    ax[1].set_xticks(range(len(order)), order, rotation=60, fontsize=8)
    ax[1].set_ylabel(f"{metric}: {label_a or arm_a} − {label_b or arm_b}"); ax[1].legend(); ax[1].grid(axis="y", alpha=0.3)
    ax[1].set_title(f"{sum(v > 0 for v in d.values())} of {len(d)} patients improve")
    fig.suptitle(f"{label_a or arm_a} vs {label_b or arm_b}, paired by patient  ({TAG})")
    plt.tight_layout(); plt.show()
    return d

A, B = RX.COMPARISONS["views"]
d_views = paired_figure(A, B, label_a="AP + LAT (2 views)", label_b="AP only (1 view)")

**Why pairs?** The spread *between* patients (the left panel's vertical spread) is usually far bigger than the change
*within* a patient (the slope of each line). Comparing two clouds of numbers drowns the effect in the patient spread;
the paired differences remove it, because every patient is compared with itself.

A standard test of "are the paired differences centred on 0?" is the **Wilcoxon signed-rank test** (SciPy's, as
plumbing: it ranks the |differences| and asks whether the positive ones carry more of the rank than chance would).
Part C builds the other tool, the one you can explain line by line: a bootstrap interval.

In [ ]:
from scipy.stats import wilcoxon
diffs = np.array(list(d_views.values()))
try:
    w = wilcoxon(diffs)
    print(f"Wilcoxon signed-rank, {len(diffs)} pairs: statistic {w.statistic:.1f}, p = {w.pvalue:.3f}  ({TAG})")
except ValueError as e:            # e.g. every difference exactly 0
    print("Wilcoxon not defined here:", e)
print(f"mean difference {diffs.mean():+.4f}, {np.sum(diffs > 0)} of {len(diffs)} patients improve")

**Explain-back (part B):**
1. `mean_std` divides by n − 1. With three seeds, how precise is the std itself? (Video 12: a rough estimate. What
   would ten seeds buy, and what would they cost on Kaggle?)
2. `seed_scores` averages over patients, `per_subject_mean` averages over seeds. Which kind of noise does each one
   show, and which one hides?
3. Why must the 1-view and 2-view arms be scored on exactly the same test patients for the pairing to mean
   anything? What would happen if one arm's CSV missed a hard patient?

**Part B done.** Move `mean_std`, `seed_scores`, `per_subject_mean` and `paired_differences` into
`src/xray3d/experiment.py`.

---
## Part C · Is it real? (after video 12: the honest table) · comparison 2: 25 % vs 100 % of the training set

A mean difference is one number from a handful of patients. **How sure are we of its sign?** The bootstrap answers by
pretending the test patients we have *are* the population: draw a new "test set" of the same size from them, with
replacement, recompute the mean difference, and repeat thousands of times. The spread of those means is how much
the result would move with a different set of patients.

### TODO C1 · `bootstrap_ci`  ·  suggested driver: **Noa**

A **percentile bootstrap** interval for the mean of the paired differences: `n_boot` times, draw `n` differences
with replacement out of the `n` you have and take their mean; the interval is the `(1 − level)/2` and
`1 − (1 − level)/2` percentiles of those means. Use `np.random.default_rng(seed)` so the interval is reproducible.

> **Nudge:** one bootstrap sample = `n` random indices in 0 … n − 1, **with** replacement (`rng.integers`). An index array of shape `(n_boot, n)` does all the samples at once; `.mean(axis=1)` gives one mean per sample; `np.percentile` takes percents (0–100), not fractions.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def bootstrap_ci(diffs, n_boot=10_000, level=0.95, seed=0):
    """Percentile bootstrap confidence interval of the mean of paired differences (resampling patients).

    diffs   : iterable of per-patient differences, e.g. paired_differences(a, b).values()
    n_boot  : number of bootstrap samples
    level   : 0.95 -> the 2.5th and 97.5th percentiles
    seed    : for np.random.default_rng
    returns : (lo, hi) as Python floats
    """
    raise NotImplementedError("TODO C1")

In [ ]:
T.check_bootstrap_ci(bootstrap_ci)

### TODO C2 · `verdict`  ·  suggested driver: **Adir**

The decision rule. A difference between arm A and arm B counts as **real** only if both hold:
1. the paired bootstrap interval (over patients) **excludes 0**, and
2. the difference of the arms' means over seeds, `mean(seeds_a) − mean(seeds_b)`, is **bigger than the seed spread**:
   larger in size than the bigger of the two arms' sample stds.

Return `+1` if A is better, `−1` if B is better, `0` for "within noise". "Better" means higher for Dice and lower for
HD95 (`higher_is_better=False`).

> **Nudge:** use your B1 for both arms. Condition 1 answers "another set of patients", condition 2 "another seed": a result has to survive both. For lower-is-better metrics compute everything the same way and flip the sign of the answer at the end.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def verdict(seeds_a, seeds_b, ci, higher_is_better=True):
    """Is arm A better than arm B, worse, or within noise?

    seeds_a, seeds_b : one score per run (seed) of each arm, e.g. seed_scores(rows, "ap_lat").values()
    ci               : (lo, hi), the bootstrap interval of the paired differences A - B
    returns          : +1 (A better), -1 (B better) or 0 (within noise)
    """
    raise NotImplementedError("TODO C2")

In [ ]:
T.check_verdict(verdict)

In [ ]:
A, B = RX.COMPARISONS["size"]
d_size = paired_figure(A, B, label_a="2 views, 100 % of train", label_b="2 views, 25 % of train")

In [ ]:
WORDS = {1: "{a} is better", -1: "{b} is better", 0: "within noise"}
def compare(name, metric="dice", higher=True):
    a_arm, b_arm = RX.COMPARISONS[name]
    sa, sb = seed_scores(rows, a_arm, metric), seed_scores(rows, b_arm, metric)
    d = paired_differences(per_subject_mean(rows, a_arm, metric), per_subject_mean(rows, b_arm, metric))
    lo, hi = bootstrap_ci(d.values())
    (ma, sda), (mb, sdb) = mean_std(sa.values()), mean_std(sb.values())
    v = verdict(sa.values(), sb.values(), (lo, hi), higher_is_better=higher)
    print(f"{name:<6} {metric:<5} A = {a_arm:<11} {ma:7.3f} ± {sda:.3f}   B = {b_arm:<11} {mb:7.3f} ± {sdb:.3f}   "
          f"A - B = {ma - mb:+.3f}, 95 % CI over {len(d)} patients [{lo:+.3f}, {hi:+.3f}]  ->  {WORDS[v].format(a=a_arm, b=b_arm)}")
    return v

print(TAG)
for name in RX.COMPARISONS:
    compare(name, "dice")
    compare(name, "hd95", higher=False)

### TODO C3 · `results_table`  ·  suggested driver: **Noa**

The final table: one row per arm and per baseline, each with `mean`, `std`, `runs` and `subjects`.
* An **arm**: `mean ± std` over its **seed scores** (B2 + B1): the ± is the seed spread. `runs` = number of seeds.
* A **baseline** (unit 03's rows: `method`, `subject`, metrics): it has no seeds (it is deterministic), so its mean is
  over its subjects, its `std` is `nan` and `runs` is 1. Writing its spread over patients there would put a
  different kind of ± in the same column.
* Every row must be measured on **the same patients** (video 12: "n = test patients, the same in every row"): if a
  baseline's subjects differ from the experiment's, raise `ValueError`.

Arms first, in the order they appear in `rows`; then the baselines, in the order they appear.

> **Nudge:** reuse your B1 and B2 for the arms. `list(dict.fromkeys(...))` keeps the order of first appearance without duplicates. Compare subject *sets*.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def results_table(rows, baseline_rows=(), metric="dice"):
    """The results table for one metric.

    rows          : the experiment's rows (keys "arm", "seed", "subject", metric)
    baseline_rows : unit 03's rows (keys "method", "subject", metric), may be empty
    returns       : list of dicts {"method", "mean", "std", "runs", "subjects"}: arms first, then baselines
    raises        : ValueError if a baseline was scored on different subjects than the experiment
    """
    raise NotImplementedError("TODO C3")

In [ ]:
T.check_results_table(results_table)

In [ ]:
TABLE_METRICS = [("dice", "Dice ↑"), ("hd95", "HD95 mm ↓"), ("nsd_5mm", "NSD 5 mm ↑")]
exp_subjects = {r["subject"] for r in rows}
use_base = [r for r in base if r["method"] != "oracle_retrieval"]      # the oracle cheats with the GT: a ceiling, not a method
if use_base and {r["subject"] for r in use_base} != exp_subjects:
    print("WARNING: baselines.csv was scored on other test subjects than the experiment (e.g. the local 27-subject "
          "dataset vs Kaggle's full one). Re-run unit 03's notebook on the same dataset. Baselines left out.")
    use_base = []
tables = {m: results_table(rows, use_base, m) for m, _ in TABLE_METRICS}
NAMES = {"ap": "network, AP only", "ap_lat": "network, AP + LAT", "ap_lat_f25": "network, AP + LAT, 25 % of train"}
cell = lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}" if np.isfinite(r["std"]) else f"{r['mean']:.3f}"
lines = [f"**{TAG}**: mean ± sample std over {len(RX.SMOKE['seeds']) if IS_SMOKE else len(RX.SEEDS)} seeds; "
         f"n = {len(exp_subjects)} test patients, the same in every row. Baselines are single deterministic runs (no ±).", "",
         "| method | runs | " + " | ".join(lbl for _, lbl in TABLE_METRICS) + " |",
         "|---|---|" + "---|" * len(TABLE_METRICS)]
for i, r in enumerate(tables["dice"]):
    lines.append(f"| {NAMES.get(r['method'], r['method'])} | {r['runs']} | " + " | ".join(cell(tables[m][i]) for m, _ in TABLE_METRICS) + " |")
cfg = trainer.DEFAULTS
n_par = {v: sum(p.numel() for p in nets.XrayTo3D(n_views=v).parameters()) for v in (1, 2)}   # as in part A's cell
lines += ["", f"Fixed in every network row: the unit 04 network ({n_par[1] / 1e3:.0f} k parameters with 1 view, "
          f"{n_par[2] / 1e3:.0f} k with 2), "
          f"{RX.SMOKE['iters'] if IS_SMOKE else RX.ITERS} steps, batch {RX.SMOKE['batch_size'] if IS_SMOKE else cfg['batch_size']}, "
          f"Adam {RX.SMOKE['lr'] if IS_SMOKE else cfg['lr']}, the split of unit 01, best checkpoint by validation Dice, test scored once."]
n_bad = sum(1 for r in rows if not np.isfinite(r["hd95"]))      # inf = empty prediction (see above), nan = failed
if n_bad:
    lines += ["", f"HD95: {n_bad} of {len(rows)} (run, subject) rows are not finite (empty prediction -> inf), "
              "so an arm's HD95 mean can be inf or nan."]
TABLE_MD = "\n".join(lines)
print(TABLE_MD)
if not IS_SMOKE:
    (env.results_root() / "experiment_v1_table.md").write_text(TABLE_MD + "\n", encoding="utf-8")
    print("\nwrote", env.results_root() / "experiment_v1_table.md")

**Explain-back (part C, and the whole unit):**
1. **Why change one variable?** What is a confound, and is the 2-view network's extra encoder one? Which way could it
   push the result, and what would the zero-LAT variant from video 12 control that ours doesn't?
2. **Why seeds?** Point at the seed dot plot: what would a single run per arm have let you claim, and how wrong
   could it have been?
3. **Why pair by patient?** Look at the paired figure: is the spread between patients bigger or smaller than the
   change within a patient? What would an unpaired comparison of the two arms' means have to overcome?
4. **"Within noise"**: what exactly does a verdict of 0 say, and what does it *not* say? (Is "no evidence of a
   difference" the same as "no difference"? What would make the interval narrower?)
5. **If two views don't help** (verdict 0 for Dice on the real run), what do you write in the report? Think of:
   the interval as the result ("the second view changed Dice by −0.01 to +0.03"), the number of patients, the
   baselines (does either arm beat nearest neighbour and the mean shape?), and what could explain it (the network
   learns a typical pelvis and the AP image only places it; 9 or ~60 test patients; 5 mm voxels).
6. Comparison 2: if 100 % beats 25 % clearly, what does that predict for the full dataset vs the small one? If it
   doesn't, what does that say about the network?

**Part C done.** Move `bootstrap_ci`, `verdict` and `results_table` into `src/xray3d/experiment.py`. Then run pytest:

In [ ]:
!{sys.executable} -m pytest -q -rs .

Fill in `LOG.md`: driver per TODO, hints used, the explain-back answers, the real table (from Kaggle) and the time each
run took. Commit `results/experiment_v1.csv` and `results/experiment_v1_table.md` (small, no data). Next: **unit 07**,
real X-rays.